In [1]:
# https://www.ncbi.nlm.nih.gov/refseq/MANE/
# https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/release_1.5/MANE.GRCh38.v1.5.ensembl_protein.faa.gz
# https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/release_1.5/MANE.GRCh38.v1.5.summary.txt.gz
# https://www.uniprot.org/database/DB-0261
#!wget https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/release_1.5/MANE.GRCh38.v1.5.ensembl_protein.faa.gz
#wget https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/release_1.5/MANE.GRCh38.v1.5.refseq_protein.faa.gz
#wget https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/release_1.5/MANE.GRCh38.v1.5.summary.txt.gz
#wget https://ftp.ncbi.nlm.nih.gov/refseq/MANE/MANE_human/release_1.5/README_versions.txt
import pandas as pd, pooled_ppi
from pooled_ppi.core import *

In [3]:
proteins = pooled_ppi.fasta.read('MANE.GRCh38.v1.5.ensembl_protein.faa')
#proteins = pooled_ppi.fasta.read('MANE.GRCh38.v1.5.refseq_protein.faa')
printlen(proteins, 'number of proteins')
proteins['seq_len'] = proteins['seq'].map(len)
proteins['is_oversized'] = proteins['seq_len'] > (5120 / 2)
proteins['is_oversized'].value_counts()

19,367 number of proteins


is_oversized
False    19114
True       253
Name: count, dtype: int64

In [4]:
#pd.read_csv('MANE.GRCh38.v1.5.summary.txt', sep='\t')['MANE_status'].value_counts()
#proteins.query('~is_oversized').sample(1000, random_state=GUARANTEED_RANDOM)[['id', 'seq_len']].to_csv('MANE_subset_1k.tsv', sep='\t', index=False, header=False)
#proteins.query('~is_oversized').sample(2000, random_state=GUARANTEED_RANDOM)[['id', 'seq_len']].to_csv('MANE_subset_2k.tsv', sep='\t', index=False, header=False)
#proteins.query('~is_oversized').sample(5000, random_state=GUARANTEED_RANDOM)[['id', 'seq_len']].to_csv('MANE_subset_5k.tsv', sep='\t', index=False, header=False)
proteins.query('~is_oversized')[['id', 'seq_len']].to_csv('MANE_not_oversized.tsv', sep='\t', index=False, header=False)

In [5]:
# 1min 30sec
#!time cat MANE_subset_1k.tsv | pooled-ppi-sample > MANE_subset_1k_pools.tsv
#!time cat MANE_subset_2k.tsv | pooled-ppi-sample > MANE_subset_2k_pools.tsv
#!time cat MANE_subset_5k.tsv | pooled-ppi-sample > MANE_subset_5k_pools.tsv

In [6]:
!time cat MANE_not_oversized.tsv | pooled-ppi-sample > MANE_not_oversized_pools.tsv

16 threads available for numba
19114 proteins in input
100%|████████████████████████| 182662941/182662941 [1:45:48<00:00, 28770.62it/s]
5752957 pools generated
182662941 interactions expected
182662941 interactions across all pools generated
True pools include all possible interactions
1.149123025375283 length-weighted redundancy factor across all pools

real	110m37.946s
user	1674m30.021s
sys	3m30.676s


In [8]:
pp = pooled_ppi.PooledPredictions(
    pools = pd.read_csv('MANE_not_oversized_pools.tsv', sep='\t'),
    sizes = pd.read_csv('MANE_not_oversized.tsv', sep='\t', names=['af3_id', 'seq_len']),
)
pp.pool_coverage()

pool_coverage: 19,114 proteins
pool_coverage: 5,752,957 pools
pool_coverage: 182,662,941 interactions possible based on the list of proteins
pool_coverage: 182,662,941 interactions in pools
pool_coverage: 0 interactions missing
pool_coverage: 0 interactions extra
